# 部署 09 · 为什么需要部署平台 + Aegra 项目骨架

这一课回答两个问题，一个「为什么」、一个「长什么样」：

1. **为什么需要一层部署平台？** —— 因为 `langgraph build` 打出来的官方 `langgraph-api`
   镜像**不是** MIT 开源的 langgraph 库，而是 LangGraph Platform（现名 LangSmith
   Deployments），容器启动时强制校验 license，过不了就直接退出。搞清「哪部分收费」，
   才知道该往哪找开源替代（Aegra，Apache 2.0）。
2. **一个真实的部署项目骨架长什么样？** —— 七个文件各自干什么、`aegra.json` 里那个
   最容易抄错的 `dependencies` 到底什么语义，然后把整套骨架**真的落盘**跑一遍冒烟导入。

| 概念 | 是什么 | 本课的代码形态 |
|---|---|---|
| license 校验 | 官方运行时镜像的启动门槛 | `LICENSE_ERROR_TEXT`（课案报错原文） |
| 库 vs 平台 | 图引擎（MIT 免费）/ 部署运行时（商业） | `print_table("开源 vs 商业…")` |
| 选型地图 | LangSmith 一个产品干两件事，可替代性不同 | `section_3_table_smith_roles` |
| 运行时四要素 | Threads API / Runs / 流式 / Checkpoint | `section_4_table_runtime_elements` |
| Aegra | LangGraph Platform 的开源自托管替代 | `AEGRA_REPO` / `section_6_aegra` |
| 项目骨架 | `aegra.json` + `my_agent/graph.py` + 容器三件套 | `SKELETON_FILES`（7 个文件） |
| `dependencies` 语义 | aegra.json 里是**加进 sys.path 的目录列表** | `section_5_aegra_json` |

> **本 notebook 由 `Agent/_py_source/09_aegra_deploy/` 下 2 个脚本合并而成**：
> `01_为什么需要部署平台_jxsd.py`（332 行，原理篇）
> + `02_项目骨架_jxsd.py`（718 行，动手篇）。
> 718 行的那篇没有变成一整格代码：它被拆成「目录树 → 逐文件讲解 → 骨架代码块 → 落盘 → 冒烟」，
> 每个文件一格，长字符串（docker-compose.yml）再拆成两格用 `+=` 续写。

**官方文档**
- LangGraph 部署总览：<https://docs.langchain.com/oss/python/langgraph/deploy>
- Aegra 仓库（Apache 2.0）：<https://github.com/aegra/aegra>

## 运行条件

| 项 | 说明 |
|---|---|
| 🟢 运行档位 | **离线可跑** —— 不起任何服务、不连大模型、不跑 Docker |
| 依赖 | 标准库为主；末尾的「冒烟导入」用到 `langgraph` / `langchain-openai` / `pydantic-settings`（venv 已装） |
| 密钥 | 不主动读取。只有冒烟导入会 import 仓库根 `config.py`（由 pydantic-settings 加载 `.env`），**不发起任何网络请求** |
| 前置服务 | 无（本课**不真起 Aegra**，只讲原理 + 静态展示骨架） |
| 预计耗时 | < 5 秒 |

> 为什么是 🟢：这一课的两件事都不需要外部服务 —— 上半节纯打印，
> 下半节只是「把文件写到本课临时目录 → 读回来 → import 一次」。
> 真正的 `aegra dev` / `aegra up` 在下一课（`02_本地开发_客户端调用_Langfuse.ipynb`）。

> ⚠️ 落盘位置：源脚本是写到脚本同级的 `aegra_project/`，那个目录**已经入库**。
> 本 notebook 改成写 `WORKDIR / "aegra_skeleton"`（临时目录），
> 仓库里那份真实骨架只作**只读展示**（见第 12 节），一个字都不动。

## 本节地图

上半节是一条「找替代方案」的推理链，下半节是「把方案落成文件」。

```mermaid
graph TD
    A["① license 坑<br/>langgraph-api 镜像启动即退出"] --> B["② 库 vs 平台<br/>到底谁收费"]
    B --> C["③ 选型地图 · 表一<br/>LangSmith 的两个角色"]
    C --> D["④ 能力清单 · 表二<br/>运行时四要素"]
    D --> E["⑤ 一句话总结<br/>怎么跑 / 跑在哪 / 结果存哪"]
    E --> F["⑥ Aegra<br/>Apache 2.0 自托管替代"]
    F --> G["⑦ 骨架七文件<br/>aegra.json / graph.py / ..."]
    G --> H["⑧ 落盘 + 冒烟导入"]
```

裸 JupyterLab 不渲染 mermaid，等价表格如下：

| 节 | 讲什么 | 对应源文件的小节 |
|---|---|---|
| 1 坑现场 | 报错原文 + 两条「官方出路」 | `01_为什么需要部署平台_jxsd.py` 第 1 节 |
| 2 收费边界 | langgraph 库 vs 官方部署平台 | 第 2 节 |
| 3 表一 | 追踪能不能用 Langfuse 顶掉 | 第 3 节 |
| 4 表二 | 运行时四要素在两种形态下分别谁提供 | 第 4 节 |
| 5 一句话总结 | 「怎么跑」vs「跑在哪 + 谁在跑 + 结果存哪」 | 第 5 节 |
| 6 Aegra | 开源自托管替代 + 四行对比表 | 第 6 节 |
| 7~9 骨架地基 | 前置条件 / 安装 / 五个命令 / 目录树 / aegra.json | `02_项目骨架_jxsd.py` 第 1~5 节 |
| 10 骨架内容 | 七个文件逐个看 | 第 6 节的 `SKELETON_FILES` |
| 11~13 落盘 | 真写文件 → 目录树 → 冒烟导入 → 下一步 | 第 6~7 节 + 主流程 |

**与上下节的衔接**：上一课（`08_*`）把 Agent 写成了一个能跑的图；
这一课解释「图能跑」和「图能被别人调用」之间差的那一层平台是什么，并把骨架搭起来；
下一课就直接在这个骨架上 `aegra dev` 把服务起起来。

## 0. 环境引导

notebook 的工作目录默认是它自己所在的文件夹，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，骨架里的 `graph.py` 在做冒烟导入时会 `ModuleNotFoundError: config`。

> 这一格顺便给出 `NB_DIR`（notebook 所在目录）与 `WORKDIR`（本课临时目录）：
> 源脚本里的 `Path(__file__).resolve().parent` 在本课全部改写成这两个变量。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

### 预期输出

```text
仓库根： F:\ProGram\Python_Base
临时目录： F:\ProGram\Python_Base\Agent\09_aegra_deploy\tmp_nb_work
```

`仓库根` 是本机实际路径（这一课所有相对路径都相对它）；`临时目录` 就是本课的 `WORKDIR`。
它落在 notebook 同级（`tmp_nb_work/`，已被 `.gitignore` 覆盖），
所以本课生成骨架、写 `.env.example` 都不会污染仓库。

### 前置条件自检

本课是 🟢 离线课，**没有「缺了就得跳过」的前置条件**，所以这一格只做「有没有」的体检，
让你知道哪些依赖是可选的：

- 前三个包只影响**最后一节的冒烟导入**（真的 import 一次生成的 `graph.py`）；
- `aegra` 命令本机没装也不影响本课 —— 本节不跑 Aegra，第 03 课才需要它；
- 不检查端口、不检查 Docker：本课不起任何容器。

In [ ]:
# ===== 前置条件自检（离线课：只报告「有没有」，没有任何东西会被跳过）=====
import importlib.util as _ilu

print("=" * 72)
print("前置条件自检")
print("=" * 72)
print(f"  Python：{sys.version.split()[0]}   （Aegra 硬性要求 3.11+）")
for _mod, _why in [
    ("langgraph", "生成的 graph.py 里 StateGraph 要用"),
    ("langchain_openai", "生成的 graph.py 里 ChatOpenAI 要用"),
    ("pydantic_settings", "仓库根 config.py 读 .env 要用"),
]:
    _ok = _ilu.find_spec(_mod) is not None
    print(f"  {'✅' if _ok else '❌'} {_mod:<18} {_why}")
_aegra_ok = _ilu.find_spec("aegra") is not None
print(f"  {'✅' if _aegra_ok else '·'} aegra-cli           {'已安装' if _aegra_ok else '本机没装 —— 本课用不到，第 03 课才跑它'}")
print()
print("  上面三个包只影响第 11 节的「冒烟导入」；缺了它会在那一格打印中文原因，")
print("  前面 1~10 节（纯讲解 + 写文件）一律照跑。")

### 预期输出

```text
========================================================================
前置条件自检
========================================================================
  Python：3.12.12   （Aegra 硬性要求 3.11+）
  ✅ langgraph          生成的 graph.py 里 StateGraph 要用
  ✅ langchain_openai   生成的 graph.py 里 ChatOpenAI 要用
  ✅ pydantic_settings  仓库根 config.py 读 .env 要用
  · aegra-cli           本机没装 —— 本课用不到，第 03 课才跑它

  上面三个包只影响第 11 节的「冒烟导入」；缺了它会在那一格打印中文原因，
  前面 1~10 节（纯讲解 + 写文件）一律照跑。
```

三个包都在，所以第 11 节的冒烟导入会走成功分支；
`aegra-cli` 那一行显示「本机没装」是**正常**的 —— 本课不跑 Aegra 服务，下一课才需要它。

## 1. 坑现场：`langgraph build` 出来的镜像启动即退出

### 1.1 先把报错原文摆出来（源文件第 1 节）

`langgraph build` 构建的官方 `langgraph-api` 镜像属于 LangGraph Platform
（现改名 LangSmith Deployments），容器启动时**强制校验 license**，验证不过直接退出。

下面这段报错在本课里**逐字保留课案原文** —— 见过它一次，以后就不会再怀疑是自己的图写错了。

先准备一个小工具 `print_table`：中文/全角字符在终端里占 2 个字符位，直接用 `len()`
算宽度会让表格歪掉，所以先用 `unicodedata.east_asian_width` 判宽度再补空格。
这一格**没有输出**，它给后面所有表格提供排版能力（两个源文件都用它）。

In [ ]:
import unicodedata


# ================================================================
# 小工具：按「显示宽度」对齐打印表格
# ================================================================
def _disp_width(text: str) -> int:
    """按东亚字符宽度计算字符串在终端里占的列数"""
    return sum(2 if unicodedata.east_asian_width(ch) in ("W", "F") else 1 for ch in text)


def _pad(text: str, width: int) -> str:
    """把 text 右侧补空格到指定显示宽度"""
    return text + " " * max(0, width - _disp_width(text))


def print_table(title: str, headers: list, rows: list) -> None:
    """打印一张带标题的等宽表格（单元格过长时不做折行，保持原貌）"""
    widths = [
        max(_disp_width(headers[i]), *(_disp_width(str(r[i])) for r in rows))
        for i in range(len(headers))
    ]
    line = "+" + "+".join("-" * (w + 2) for w in widths) + "+"
    print(f"\n【{title}】")
    print(line)
    print("| " + " | ".join(_pad(str(headers[i]), widths[i]) for i in range(len(headers))) + " |")
    print(line)
    for row in rows:
        print("| " + " | ".join(_pad(str(row[i]), widths[i]) for i in range(len(row))) + " |")
    print(line)

现在复现那段报错，并把报文里给出的两条「官方出路」翻译成人话：
两条都要求**注册或付费**，所以它不是「配置写错了」，而是「镜像本身就要 license」。

In [ ]:
print("Agent 课案 · 部署 ①：为什么需要部署平台（LangSmith Deployments 的 license 坑 → Aegra）")

# ================================================================
# 1. 课案原文：license 校验失败的报错
# ================================================================
LICENSE_ERROR_TEXT = """ValueError: License verification failed. Please ensure proper configuration:
- For local development, set a valid LANGSMITH_API_KEY for an account with LangGraph Cloud access.
- For production, configure the LANGGRAPH_CLOUD_LICENSE_KEY environment variable."""


def section_1_license_error() -> None:
    """复现课案里那段 license 报错长什么样，并解释两条出路"""
    print("=" * 78)
    print("1. 课案原文：官方 langgraph-api 镜像启动时的 license 校验失败")
    print("=" * 78)
    print("容器启动时校验不过，进程直接退出，日志里就是这一段：\n")
    for ln in LICENSE_ERROR_TEXT.splitlines():
        print("    " + ln)

    # 两条「官方出路」都指向付费/注册，不是配置错误：
    #   · 本地开发：注册 LangSmith 账号 → 配 LANGSMITH_API_KEY
    #     （账号还必须具备 LangGraph Cloud 访问权限）
    #   · 生产环境：企业版付费 license → 配 LANGGRAPH_CLOUD_LICENSE_KEY
    print("\n报文里给出的两条出路（都要求注册 / 付费，不是配置错误那么简单）：")
    print("    · 本地开发：注册 LangSmith 账号并配置 LANGSMITH_API_KEY")
    print("      （账号还必须具备 LangGraph Cloud 访问权限）")
    print("    · 生产环境：需要企业版付费 license，配置 LANGGRAPH_CLOUD_LICENSE_KEY")
    print("\n>>> 结论：这不是你代码写错了，是镜像本身要求 license。")


section_1_license_error()

### 预期输出

```text
Agent 课案 · 部署 ①：为什么需要部署平台（LangSmith Deployments 的 license 坑 → Aegra）
==============================================================================
1. 课案原文：官方 langgraph-api 镜像启动时的 license 校验失败
==============================================================================
容器启动时校验不过，进程直接退出，日志里就是这一段：

    ValueError: License verification failed. Please ensure proper configuration:
    - For local development, set a valid LANGSMITH_API_KEY for an account with LangGraph Cloud access.
    - For production, configure the LANGGRAPH_CLOUD_LICENSE_KEY environment variable.

报文里给出的两条出路（都要求注册 / 付费，不是配置错误那么简单）：
    · 本地开发：注册 LangSmith 账号并配置 LANGSMITH_API_KEY
      （账号还必须具备 LangGraph Cloud 访问权限）
    · 生产环境：需要企业版付费 license，配置 LANGGRAPH_CLOUD_LICENSE_KEY

>>> 结论：这不是你代码写错了，是镜像本身要求 license。
```

## 2. 澄清误解：到底哪部分是收费的

把「库」和「平台」拆开看，收费边界立刻清楚：

| 东西 | 是否开源 | 说明 |
|---|---|---|
| langgraph（pip 包） | MIT 开源，免费 | 图引擎本体：StateGraph / 节点 / 边 / 中断 |
| langgraph-checkpoint-* | MIT 开源，免费 | 检查点存储（内存 / SQLite / PostgreSQL） |
| langgraph-sdk | MIT 开源，免费 | 客户端 SDK（连官方平台和连 Aegra 用的是同一个） |
| langgraph-api 镜像 / LangSmith Deployments | 商业产品 | 托管运行时 + REST API + 控制台 + 多租户 |

一句话：**你写的 Agent 代码是你的，跑 Agent 的那层「平台」才是商品。**
既然那层平台对外的契约只是「一套 Agent Protocol 的 HTTP API」，
就完全可以自己用 FastAPI + PostgreSQL 复刻 —— 这就是 Aegra 做的事。

In [ ]:
# ================================================================
# 2. 澄清一个高频误解：到底哪部分是收费的？
# ================================================================
def section_2_what_is_paid() -> None:
    print("\n" + "=" * 78)
    print("2. 到底哪部分收费？—— 库 vs 平台")
    print("=" * 78)
    print_table(
        "开源 vs 商业：逐项拆开看",
        ["组成部分", "许可", "说明"],
        [
            ["langgraph（pip 包）", "MIT 开源", "图引擎本体：StateGraph / 节点 / 边 / 中断"],
            ["langgraph-checkpoint-*", "MIT 开源", "检查点存储：内存 / SQLite / PostgreSQL"],
            ["langgraph-sdk", "MIT 开源", "客户端 SDK（连官方平台与连 Aegra 是同一个）"],
            ["langgraph-api 镜像", "商业产品", "LangGraph Platform / LangSmith Deployments"],
            ["LangSmith Deployments", "商业产品", "托管运行时 + REST API + 控制台 + 多租户"],
        ],
    )
    print("\n>>> 你写的 Agent 代码是你的；跑 Agent 的那层「平台」才是商品。")


section_2_what_is_paid()

### 预期输出

```text

==============================================================================
2. 到底哪部分收费？—— 库 vs 平台
==============================================================================

【开源 vs 商业：逐项拆开看】
+------------------------+----------+---------------------------------------------+
| 组成部分               | 许可     | 说明                                        |
+------------------------+----------+---------------------------------------------+
| langgraph（pip 包）    | MIT 开源 | 图引擎本体：StateGraph / 节点 / 边 / 中断   |
| langgraph-checkpoint-* | MIT 开源 | 检查点存储：内存 / SQLite / PostgreSQL      |
| langgraph-sdk          | MIT 开源 | 客户端 SDK（连官方平台与连 Aegra 是同一个） |
| langgraph-api 镜像     | 商业产品 | LangGraph Platform / LangSmith Deployments  |
| LangSmith Deployments  | 商业产品 | 托管运行时 + REST API + 控制台 + 多租户     |
+------------------------+----------+---------------------------------------------+

>>> 你写的 Agent 代码是你的；跑 Agent 的那层「平台」才是商品。
```

（上面表格的竖线对齐由 `_disp_width` 按**显示宽度**算出来，所以中文列也是齐的。）

## 3. 选型地图 · 表一：LangSmith 的两个角色，Langfuse 能替代吗

这张表是整章的「选型地图」：LangSmith 其实**一个产品干了两件事**，
而这两件事的可替代性完全不同 ——

| LangSmith 的角色 | Langfuse 能替代吗 | 开源替代 |
|---|---|---|
| 观测/追踪（trace、调试、评估） | ✅ 能，Langfuse 的主业（MIT 开源） | Langfuse |
| 部署运行时（threads API、runs、流式、checkpoint 管理） | ❌ 不能，Langfuse 完全不做这块 | Aegra、或自己用 FastAPI 包一层 |

关键结论：「我上了 Langfuse 是不是就不用管部署运行时了」——**这个念头是错的**。
Langfuse 是观测平台，不是运行时平台；运行时那块要靠 Aegra（或自己包一层）来补。

In [ ]:
# ================================================================
# 3. 表一：LangSmith 的两个角色，Langfuse 能替代吗？
# ================================================================
def section_3_table_smith_roles() -> None:
    print("\n" + "=" * 78)
    print("3. 表一：LangSmith 的角色与替代方案")
    print("=" * 78)
    print_table(
        "LangSmith 的角色与替代",
        ["LangSmith 的角色", "Langfuse 能替代吗", "开源替代"],
        [
            ["观测/追踪（trace、调试、评估）", "✅ 能，Langfuse 的主业（MIT 开源）", "Langfuse"],
            ["部署运行时（threads API、runs、流式、checkpoint 管理）",
             "❌ 不能，Langfuse 完全不做这块", "Aegra、或自己用 FastAPI 包一层"],
        ],
    )
    print("\n>>> 两个角色要分开选型：追踪用 Langfuse，运行时用 Aegra，两者互不冲突。")


section_3_table_smith_roles()

### 预期输出

```text

==============================================================================
3. 表一：LangSmith 的角色与替代方案
==============================================================================

【LangSmith 的角色与替代】
+--------------------------------------------------------+------------------------------------+--------------------------------+
| LangSmith 的角色                                       | Langfuse 能替代吗                  | 开源替代                       |
+--------------------------------------------------------+------------------------------------+--------------------------------+
| 观测/追踪（trace、调试、评估）                         | ✅ 能，Langfuse 的主业（MIT 开源） | Langfuse                       |
| 部署运行时（threads API、runs、流式、checkpoint 管理） | ❌ 不能，Langfuse 完全不做这块     | Aegra、或自己用 FastAPI 包一层 |
+--------------------------------------------------------+------------------------------------+--------------------------------+

>>> 两个角色要分开选型：追踪用 Langfuse，运行时用 Aegra，两者互不冲突。
```

表里两个单元格的 emoji 都是**从课案原文照抄**的，课程后半段（`05_对接Langfuse`）
会真的把「追踪」那一行落到 Langfuse 上。

## 4. 能力清单 · 表二：部署运行时四要素

右边这些能力业务上**一个都不能少**，区别只是「自己拼零件」还是「平台给成一套 REST API」：

| 概念 | 说明 | LangGraph 库 vs LangSmith |
|---|---|---|
| Threads API | 对话会话管理，一个用户对话 = 一个 thread | 库：用 `thread_id` 区分 / LangSmith：REST API 建查删线程 |
| Runs | 每次 `graph.invoke()` 产生一条执行记录 | 库：同步执行 / LangSmith：异步执行 + 持久化 |
| 流式 | 实时推送中间步骤给前端 | 库：`stream_mode="updates"` / LangSmith：转 HTTP 流式响应 |
| Checkpoint 管理 | 状态快照，支持中断恢复与时间旅行 | 库：自己搭数据库 / LangSmith：托管 + 持久化 + 多租户 |

下面这一格为了在窄终端里看得清，把宽表**拆成四段**逐个展开讲。

In [ ]:
# ================================================================
# 4. 表二：部署运行时四要素，两种形态下分别怎么做
# ================================================================
def section_4_table_runtime_elements() -> None:
    print("\n" + "=" * 78)
    print("4. 表二：部署运行时四要素，两种形态下分别怎么做")
    print("=" * 78)

    rows = [
        ["Threads API", "对话会话管理。每个用户对话是一个 thread，同一 thread 内 Agent 记住上下文",
         "LangGraph 库：用 thread_id 区分\nLangSmith：提供 REST API 创建/查询/删除线程"],
        ["Runs", "执行记录。每次 graph.invoke() 产生一个 run，记录输入/输出/状态",
         "LangGraph 库：同步执行\nLangSmith：异步执行 + 持久化每次 run 的结果"],
        ["流式", "Agent 执行时实时推送中间步骤（工具调用、思考过程）给前端",
         "LangGraph 库：stream_mode=\"updates\"\nLangSmith：转为 HTTP 流式响应（text/event-stream）"],
        ["Checkpoint 管理", "状态快照。每步执行后自动保存 state，支持中断恢复和时间旅行",
         "LangGraph 库：自己搭数据库\nLangSmith：① 云端托管 ② 持久化 ③ 多租户"],
    ]

    # 四要素逐个展开讲，避免上面那张宽表在窄终端里被折行看不清
    for name, desc, cmp_ in rows:
        print(f"\n  ◆ {name}")
        print(f"      是什么：{desc}")
        for ln in cmp_.splitlines():
            print(f"      怎么做：{ln}")

    print("\n>>> 左边这些能力业务上一个都不能少；区别只是「自己拼」还是「平台给」。")


section_4_table_runtime_elements()

### 预期输出

```text

==============================================================================
4. 表二：部署运行时四要素，两种形态下分别怎么做
==============================================================================

  ◆ Threads API
      是什么：对话会话管理。每个用户对话是一个 thread，同一 thread 内 Agent 记住上下文
      怎么做：LangGraph 库：用 thread_id 区分
      怎么做：LangSmith：提供 REST API 创建/查询/删除线程

  ◆ Runs
      是什么：执行记录。每次 graph.invoke() 产生一个 run，记录输入/输出/状态
      怎么做：LangGraph 库：同步执行
      怎么做：LangSmith：异步执行 + 持久化每次 run 的结果

  ◆ 流式
      是什么：Agent 执行时实时推送中间步骤（工具调用、思考过程）给前端
      怎么做：LangGraph 库：stream_mode="updates"
      怎么做：LangSmith：转为 HTTP 流式响应（text/event-stream）

  ◆ Checkpoint 管理
      是什么：状态快照。每步执行后自动保存 state，支持中断恢复和时间旅行
      怎么做：LangGraph 库：自己搭数据库
      怎么做：LangSmith：① 云端托管 ② 持久化 ③ 多租户

>>> 左边这些能力业务上一个都不能少；区别只是「自己拼」还是「平台给」。
```

注意最后一项 Checkpoint：**库形态下它是最费事的一项**（要自己搭库、自己管多租户），
这正好解释了「为什么大家愿意为一个部署平台付费」。

## 5. 一句话总结：三个问句对应三件事

| 问句 | 对应什么 | 谁负责 |
|---|---|---|
| 怎么跑 | 图引擎 | `langgraph` 库（MIT，免费） |
| 跑在哪 | 部署形态（容器 / 服务器 / 托管） | 部署平台（商业，或用 Aegra 替代） |
| 谁在跑 | 多租户与鉴权（谁的 thread、谁的 run） | 部署平台 |
| 结果存哪 | 持久化（PostgreSQL checkpoint + runs 记录） | 部署平台 / 你自己搭 |

In [ ]:
# ================================================================
# 5. 一句话总结
# ================================================================
ONE_SENTENCE = (
    "LangGraph 负责「怎么跑」（图引擎），LangSmith 负责「跑在哪 + 谁在跑 + 跑的结果存哪」"
    "（运行时平台）。Aegra 把这个运行时平台用开源替代了。"
)


def section_5_summary() -> None:
    print("\n" + "=" * 78)
    print("5. 一句话总结")
    print("=" * 78)
    print("  " + ONE_SENTENCE)


section_5_summary()

### 预期输出

```text

==============================================================================
5. 一句话总结
==============================================================================
  LangGraph 负责「怎么跑」（图引擎），LangSmith 负责「跑在哪 + 谁在跑 + 跑的结果存哪」（运行时平台）。Aegra 把这个运行时平台用开源替代了。
```

## 6. 出路：开源替代 Aegra（Apache 2.0）

这段是整章的转折点，三个细节值得单独拎出来：

1. **同一套 Agent Protocol** —— 协议层兼容，不是「类似」而是「同一套」；
2. **`langgraph_sdk` 客户端代码原样可用** —— 以后从 Aegra 迁回官方平台（或反过来），
   客户端那几十行**一行都不用改**，只换 `url`。下一课 `04_客户端调用` 能直接照抄课案，
   靠的就是这一条；
3. **没有任何 license 校验** —— 对应第 1 节那个 `ValueError`，Aegra 里不存在。

课案原样的四行对比表（也在这里打印出来）：

| 对比 | LangSmith Deployments | Aegra |
|---|---|---|
| 自托管 | 仅企业版（license key） | 免费（Apache 2.0） |
| 数据库 | 官方托管 | 自己的 PostgreSQL |
| 追踪 | 仅 LangSmith | Langfuse |
| 客户端 SDK | LangGraph SDK | 同款 LangGraph SDK |

注意最后一行：表格里唯一「完全一样」的一格，恰恰是学员最关心的那一格 —— 迁移成本。

In [ ]:
# ================================================================
# 6~7. 开源替代：Aegra
# ================================================================
AEGRA_REPO = "https://github.com/aegra/aegra"
AEGRA_LICENSE = "Apache 2.0"


def section_6_aegra() -> None:
    print("\n" + "=" * 78)
    print("6. 开源替代：Aegra")
    print("=" * 78)
    print(f"  仓库地址：{AEGRA_REPO}  （{AEGRA_LICENSE}）")
    print("  它是什么：LangGraph Platform 的开源自托管替代。")
    print("            用 FastAPI + PostgreSQL 实现同一套 Agent Protocol，")
    print("            langgraph_sdk 客户端代码原样可用，没有任何 license 校验。")
    print("\n  三个关键细节：")
    print("    ① 「同一套」Agent Protocol —— 协议层兼容，不是「类似」。")
    print("    ② langgraph_sdk 客户端代码原样可用 —— 迁回官方平台只换 url，客户端不用改。")
    print("    ③ 没有 license 校验 —— 本节开头那个 ValueError 在 Aegra 里不存在。")

    print_table(
        "Aegra vs LangSmith Deployments（课案四行对比表）",
        ["对比", "LangSmith Deployments", "Aegra"],
        [
            ["自托管", "仅企业版（license key）", "免费（Apache 2.0）"],
            ["数据库", "官方托管", "自己的 PostgreSQL"],
            ["追踪", "仅 LangSmith", "Langfuse"],
            ["客户端 SDK", "LangGraph SDK", "同款 LangGraph SDK"],
        ],
    )
    # 最后一行是整张表唯一「完全一样」的一格，也正是学员最关心的迁移成本
    print("\n>>> 注意最后一行：两边客户端 SDK 是同款，所以迁移成本几乎为零。")
    print(">>> 下一节（02_项目骨架）就开始动手搭一个 Aegra 项目。")


section_6_aegra()
print("\n（本节纯讲解，无需任何外部服务；下一节开始生成真实项目骨架）")

### 预期输出

```text

==============================================================================
6. 开源替代：Aegra
==============================================================================
  仓库地址：https://github.com/aegra/aegra  （Apache 2.0）
  它是什么：LangGraph Platform 的开源自托管替代。
            用 FastAPI + PostgreSQL 实现同一套 Agent Protocol，
            langgraph_sdk 客户端代码原样可用，没有任何 license 校验。

  三个关键细节：
    ① 「同一套」Agent Protocol —— 协议层兼容，不是「类似」。
    ② langgraph_sdk 客户端代码原样可用 —— 迁回官方平台只换 url，客户端不用改。
    ③ 没有 license 校验 —— 本节开头那个 ValueError 在 Aegra 里不存在。

【Aegra vs LangSmith Deployments（课案四行对比表）】
+------------+-------------------------+--------------------+
| 对比       | LangSmith Deployments   | Aegra              |
+------------+-------------------------+--------------------+
| 自托管     | 仅企业版（license key） | 免费（Apache 2.0） |
| 数据库     | 官方托管                | 自己的 PostgreSQL  |
| 追踪       | 仅 LangSmith            | Langfuse           |
| 客户端 SDK | LangGraph SDK           | 同款 LangGraph SDK |
+------------+-------------------------+--------------------+

>>> 注意最后一行：两边客户端 SDK 是同款，所以迁移成本几乎为零。
>>> 下一节（02_项目骨架）就开始动手搭一个 Aegra 项目。

（本节纯讲解，无需任何外部服务；下一节开始生成真实项目骨架）
```

## 7. 第二课：Aegra 项目骨架的「地基」

从这里开始是源文件 `02_项目骨架_jxsd.py`（718 行）的内容。它讲的是「动手之前的地基」，
分三块：**环境要求与安装 → 五个常用命令 → 项目骨架里每个文件是干什么的**。

### 7.1 先定位：骨架写到哪、读哪

源脚本里这三行决定了「往哪写文件」，在 notebook 里必须改写（`__file__` 在 notebook 里不存在）：

| 源脚本 | 本 notebook | 为什么 |
|---|---|---|
| `HERE = Path(__file__).resolve().parent` | `HERE = NB_DIR` | notebook 没有 `__file__`，`NB_DIR` 就是 notebook 所在目录（`Agent/09_aegra_deploy`） |
| `PROJECT_DIR = HERE / "aegra_project"` | `PROJECT_DIR = WORKDIR / "aegra_skeleton"` | `aegra_project/` **已经入库**，不能覆盖；生成物一律进本课临时目录 |
| `REPO_ROOT = HERE.parents[1]` | 原样保留 | `HERE` 仍然在 `Agent/09_aegra_deploy`，`parents[1]` 正好是仓库根，语义不变 |

另外记一下 `REAL_PROJECT_DIR`：仓库里那份**已入库**的真实骨架，本课只读展示它。

In [ ]:
print("Agent 课案 · 部署 ②：Aegra 项目骨架（前置条件 / 安装 / 项目结构 / aegra.json）")

import importlib
import shutil

# 本 notebook 所在目录 = 09_aegra_deploy/；骨架生成到它下面的临时目录里
HERE = NB_DIR
PROJECT_DIR = WORKDIR / "aegra_skeleton"
# 仓库里已入库的真实骨架：本课只读展示，绝不写入
REAL_PROJECT_DIR = NB_DIR / "aegra_project"
# HERE = <仓库>\Agent\09_aegra_deploy
#        parents[0] = Agent\  →  parents[1] = <仓库根>
# 写错这一层，下面的冒烟导入就会找不到 config 模块。
REPO_ROOT = HERE.parents[1]

print(f"生成目标（临时）：{PROJECT_DIR}")
print(f"只读展示（已入库）：{REAL_PROJECT_DIR}")
print(f"仓库根：{REPO_ROOT}")

### 预期输出

```text
Agent 课案 · 部署 ②：Aegra 项目骨架（前置条件 / 安装 / 项目结构 / aegra.json）
生成目标（临时）：F:\ProGram\Python_Base\Agent\09_aegra_deploy\tmp_nb_work\aegra_skeleton
只读展示（已入库）：F:\ProGram\Python_Base\Agent\09_aegra_deploy\aegra_project
仓库根：F:\ProGram\Python_Base
```

三行路径都是本机的绝对路径，对照着看两个目录的分工：
`生成目标` 是本课临时目录（写），`只读展示` 是已入库的真实骨架（读），
`仓库根` 是 `HERE.parents[1]` 推出来的 —— 也就是冒烟导入要 `import config` 的那一层。

### 7.2 前置条件：Python 3.11+ / Docker / Redis

课案原表：

| 要求 | 说明 |
|---|---|---|
| Python 3.11+ | Aegra 硬性要求 |
| Docker | `aegra dev` / `aegra up` 会自动拉起 PostgreSQL（pgvector/pgvector:pg18） |
| Redis | 仅生产模式 `aegra up` 使用，也是自动拉起 |

⚠️ 这里最容易误读的一句是「Docker 和 Redis 是**自动拉起**」——
指的是「**Aegra 帮你 `docker run`**」，不是「不需要 Docker」：
本机必须装了 Docker 并且引擎在运行（Windows 上就是 Docker Desktop 开着）。
本课不起服务，所以这两条只作信息展示。

In [ ]:
# ================================================================
# 1. 前置条件
# ================================================================
def section_1_prerequisites() -> None:
    print("=" * 78)
    print("1. 前置条件")
    print("=" * 78)
    print_table(
        "前置条件",
        ["要求", "说明"],
        [
            ["Python 3.11+", "Aegra 硬性要求（本项目课案用 3.12）"],
            ["Docker", "aegra dev / aegra up 会自动拉起 PostgreSQL（pgvector/pgvector:pg18）"],
            ["Redis", "仅生产模式 aegra up 使用，也是自动拉起"],
        ],
    )
    print("\n>>> Docker / Redis 都是「自动拉起」，不用自己 docker run 起数据库。")
    print(">>> 但本机必须装了 Docker 并且 Docker 引擎处于运行状态（Windows 上就是 Docker Desktop 开着）。")


section_1_prerequisites()

### 预期输出

```text
==============================================================================
1. 前置条件
==============================================================================

【前置条件】
+--------------+----------------------------------------------------------------------+
| 要求         | 说明                                                                 |
+--------------+----------------------------------------------------------------------+
| Python 3.11+ | Aegra 硬性要求（本项目课案用 3.12）                                  |
| Docker       | aegra dev / aegra up 会自动拉起 PostgreSQL（pgvector/pgvector:pg18） |
| Redis        | 仅生产模式 aegra up 使用，也是自动拉起                               |
+--------------+----------------------------------------------------------------------+

>>> Docker / Redis 都是「自动拉起」，不用自己 docker run 起数据库。
>>> 但本机必须装了 Docker 并且 Docker 引擎处于运行状态（Windows 上就是 Docker Desktop 开着）。
```

### 7.3 安装：为什么课案要单独建一个环境

课案原文（新建 3.12 环境，三行）：

```text
conda create -n aegra python=3.12 -y
conda activate aegra
pip install aegra-cli aegra-api langgraph langchain-openai pydantic-settings
```

三点说明：

- 课案用 conda 建**独立环境**，是因为 Aegra 的依赖（尤其 `langgraph` 版本）和你现有项目
  未必兼容 —— 隔离环境最省事；
- 三个包分工不同：`aegra-cli` 是命令行工具（init/dev/up/down/serve），
  `aegra-api` 是运行时（FastAPI 服务本体），其余是 Agent 要用到的库；
- 本仓库的默认做法是 uv，等价写法是 `uv venv --python 3.12` +
  `uv pip install aegra-cli aegra-api langgraph langchain-openai pydantic-settings`。

In [ ]:
# ================================================================
# 2. 安装命令
# ================================================================
INSTALL_COMMANDS = [
    "conda create -n aegra python=3.12 -y",
    "conda activate aegra",
    "pip install aegra-cli aegra-api langgraph langchain-openai pydantic-settings",
]


def section_2_install() -> None:
    print("\n" + "=" * 78)
    print("2. 安装（新建 3.12 环境）")
    print("=" * 78)
    for cmd in INSTALL_COMMANDS:
        print("    " + cmd)
    print("\n  为什么单独建环境：Aegra 对 langgraph 版本有要求，")
    print("  塞进现有项目环境容易和已有依赖打架，隔离最省事。")
    print("  三个包的分工：aegra-cli = 命令行；aegra-api = 服务本体；其余是 Agent 用到的库。")
    print("  用 uv 的话等价写法：")
    print("    uv venv --python 3.12")
    print("    uv pip install aegra-cli aegra-api langgraph langchain-openai pydantic-settings")


section_2_install()

### 预期输出

```text

==============================================================================
2. 安装（新建 3.12 环境）
==============================================================================
    conda create -n aegra python=3.12 -y
    conda activate aegra
    pip install aegra-cli aegra-api langgraph langchain-openai pydantic-settings

  为什么单独建环境：Aegra 对 langgraph 版本有要求，
  塞进现有项目环境容易和已有依赖打架，隔离最省事。
  三个包的分工：aegra-cli = 命令行；aegra-api = 服务本体；其余是 Agent 用到的库。
  用 uv 的话等价写法：
    uv venv --python 3.12
    uv pip install aegra-cli aegra-api langgraph langchain-openai pydantic-settings
```

### 7.4 五个常用命令（其中 `aegra serve` 在 Windows 上跑不了）

课案原表（这里额外加了「补充」一列，把每条的注意点写清楚）：

| 命令 | 说明 | 补充 |
|---|---|---|
| `aegra init` | 生成模板项目（simple-chatbot / react-agent） | 交互式选模板，脚手架直接给你一个能跑的项目 |
| `aegra dev` | 本地开发：自动拉起 PostgreSQL + 热重载 | 改代码不用重启，下一课详细讲它自动做的四件事 |
| `aegra up` | 生产部署：PostgreSQL + Redis + 应用全部容器化启动 | 在服务器上执行；等价于 `docker compose up --build` |
| `aegra down` | 停止容器（加 `--volumes` 连数据卷一起删） | ⚠️ `--volumes` 会删数据卷 = 检查点/线程数据全没了 |
| `aegra serve` | 只启动应用，数据库自备。**Windows 上跑不了** | 生产走 Docker / Linux；Windows 本地开发请用 `aegra dev` |

`aegra serve` 在 Windows 上跑不了的原因是它依赖 Unix 的进程/信号模型
（Gunicorn/uvicorn worker 管理、SIGTERM 优雅退出那一套）——
**这不是配置问题，换参数也解决不了**，本地开发统一用 `aegra dev`。

In [ ]:
# ================================================================
# 3. 五个常用命令
# ================================================================
COMMANDS_TABLE = [
    ["aegra init", "生成模板项目（simple-chatbot / react-agent）",
     "交互式选择模板，脚手架直接给你一个能跑的项目"],
    ["aegra dev", "本地开发：自动拉起 PostgreSQL + 热重载",
     "改代码不用重启，下一节 03 详细讲它自动做的四件事"],
    ["aegra up", "生产部署：PostgreSQL + Redis + 应用全部容器化启动",
     "在服务器上执行；等价于 docker compose up --build"],
    ["aegra down", "停止容器（加 --volumes 连数据卷一起删）",
     "注意 --volumes 会删数据卷 = 检查点/线程数据全没了"],
    ["aegra serve", "只启动应用，数据库自备。**Windows 上跑不了**",
     "生产走 Docker / Linux；Windows 本地开发请用 aegra dev"],
]


def section_3_commands() -> None:
    print("\n" + "=" * 78)
    print("3. 五个常用命令")
    print("=" * 78)
    print_table("aegra 常用命令", ["命令", "说明", "补充"], COMMANDS_TABLE)
    # aegra serve 在 Windows 上跑不了（依赖 Unix 进程/信号模型），换参数也解决不了
    print("\n>>> 重点：aegra serve 在 Windows 上跑不了（依赖 Unix 的进程/信号模型），")
    print("    本地开发一律用 aegra dev；生产部署在 Linux 上跑 Docker。")


section_3_commands()

### 预期输出

```text

==============================================================================
3. 五个常用命令
==============================================================================

【aegra 常用命令】
+-------------+---------------------------------------------------+-------------------------------------------------------+
| 命令        | 说明                                              | 补充                                                  |
+-------------+---------------------------------------------------+-------------------------------------------------------+
| aegra init  | 生成模板项目（simple-chatbot / react-agent）      | 交互式选择模板，脚手架直接给你一个能跑的项目          |
| aegra dev   | 本地开发：自动拉起 PostgreSQL + 热重载            | 改代码不用重启，下一节 03 详细讲它自动做的四件事      |
| aegra up    | 生产部署：PostgreSQL + Redis + 应用全部容器化启动 | 在服务器上执行；等价于 docker compose up --build      |
| aegra down  | 停止容器（加 --volumes 连数据卷一起删）           | 注意 --volumes 会删数据卷 = 检查点/线程数据全没了     |
| aegra serve | 只启动应用，数据库自备。**Windows 上跑不了**      | 生产走 Docker / Linux；Windows 本地开发请用 aegra dev |
+-------------+---------------------------------------------------+-------------------------------------------------------+

>>> 重点：aegra serve 在 Windows 上跑不了（依赖 Unix 的进程/信号模型），
    本地开发一律用 aegra dev；生产部署在 Linux 上跑 Docker。
```

## 8. 项目目录树：课案原样 + 本项目的三处偏差

课案原文的目录树（`my_agent_project/`）：

```text
my_agent_project/
├── aegra.json           # 部署配置（格式兼容 langgraph.json）
├── .env                 # 环境变量
├── conf.py              # 配置文件
├── Dockerfile           # 生产镜像（aegra up 用）
├── requirements.txt
└── my_agent/
    └── graph.py         # 你的 Graph 定义
```

⚠️ 本仓库**不照抄**，有三处偏差（原因见下面的字符串常量）：

1. 用 `.env.example` 而不是 `.env` —— 模板进仓库、真 `.env` 不进 Git（里面是真的密钥）；
2. **不生成 `conf.py`**，也不另建第二套 Settings —— 大模型/数据库配置全仓库只有一份来源：
   Python_Base 根目录的 `config.py` + `.env`；
3. 本目录的 `.env.example` 只放 docker-compose 要用的「容器/端口」变量
   （compose 只读同目录的 `.env`），其余 key 一个都不重复定义。

> 顺带一个课案笔误：课案的 `my_agent/graph.py` 里写 `from config import setting`，
> 但文件名叫 `conf.py`、导出的实例叫 `settings` —— **模块名和变量名两处都对不上**。
> 本课下面会把课案的 `conf.py` 原文**打印出来对照**，但**不落盘、不参与运行**。

In [ ]:
# ================================================================
# 4. 项目目录树（课案原样）
# ================================================================
COURSE_TREE = """my_agent_project/
├── aegra.json           # 部署配置（格式兼容 langgraph.json）
├── .env                 # 环境变量
├── conf.py              # 配置文件
├── Dockerfile           # 生产镜像（aegra up 用）
├── requirements.txt
└── my_agent/
    └── graph.py         # 你的 Graph 定义"""

COURSE_TREE_NOTE = """本项目生成的骨架与课案的三处偏差：
  1. 用 .env.example 而不是 .env —— 模板进仓库、真 .env 不进 Git（里面是真的密钥）
  2. **不生成 conf.py**，也不另建第二套 Settings —— 大模型/数据库配置全仓库只有一份来源：
     Python_Base 根目录的 config.py + .env；graph.py 会自己把根目录插进 sys.path
  3. 本目录的 .env.example 只放 docker-compose 要用的「容器/端口」变量
     （compose 只读同目录的 .env），其余 key 一个都不重复定义"""

下面这个 `CONF_PY` 是**课案原文对照**：它只被打印出来给你看，**不落盘**。

为什么不落盘：全仓库一套配置来源（Python_Base 根目录 `config.py` / `.env`），
不另建第二套 `Settings` 类。注意里面还保留了课案那句笔误的注记。

In [ ]:
# 课案原样的 conf.py —— **只打印给你看，不落盘**。
# 全仓库一套配置来源（Python_Base 根目录 config.py / .env），不另建第二套 Settings 类。
CONF_PY = '''# -*- coding: utf-8 -*-
"""【课案原文对照 · 本仓库不采用】课案原样的 conf.py
===============================================================
课案的部署项目里，配置放在项目根的 conf.py，由 graph.py 去 import。

⚠️ 本项目**不用**这个文件，统一用 Python_Base 根目录的 config.py：
       from config import settings
   理由：全仓库一套配置来源，避免每个子项目各自维护一份 .env 和 Settings 类。

另外课案 my_agent/graph.py 里写的是 `from config import setting`，
和这个文件对不上（文件名是 conf.py，类实例叫 settings），是课案笔误。

import os
from pydantic import ConfigDict
from pydantic_settings import BaseSettings

PATH = os.path.dirname(os.path.abspath(__file__))


class Settings(BaseSettings):
    api_key: str
    model_name: str
    base_url: str

    mysql_user: str
    mysql_password: str
    mysql_database: str
    mysql_host: str
    mysql_port: int
    baidu_qfan_api_key: str
    gitee_api_key: str
    dashscope_api_key: str
    dashscope_base_url: str
    langfuse_secret_key: str
    langfuse_public_key: str
    langfuse_host: str

    model_config = ConfigDict(
        extra="allow",
        env_file=f"{PATH}/.env",
        case_sensitive=False,
    )


settings = Settings()
""" '''

In [ ]:
# ================================================================
# 4. 项目目录树（课案原样）
# ================================================================
def section_4_tree() -> None:
    print("\n" + "=" * 78)
    print("4. 项目目录树（课案原样）")
    print("=" * 78)
    for ln in COURSE_TREE.splitlines():
        print("    " + ln)
    print()
    for ln in COURSE_TREE_NOTE.splitlines():
        print("    " + ln)
    print("\n    ---- 课案原样的 conf.py（只打印对照，本项目不生成、不使用）----")
    for ln in CONF_PY.rstrip().splitlines():
        print("    " + ln)


section_4_tree()

### 预期输出

```text

==============================================================================
4. 项目目录树（课案原样）
==============================================================================
    my_agent_project/
    ├── aegra.json           # 部署配置（格式兼容 langgraph.json）
    ├── .env                 # 环境变量
    ├── conf.py              # 配置文件
    ├── Dockerfile           # 生产镜像（aegra up 用）
    ├── requirements.txt
    └── my_agent/
        └── graph.py         # 你的 Graph 定义

    本项目生成的骨架与课案的三处偏差：
      1. 用 .env.example 而不是 .env —— 模板进仓库、真 .env 不进 Git（里面是真的密钥）
      2. **不生成 conf.py**，也不另建第二套 Settings —— 大模型/数据库配置全仓库只有一份来源：
         Python_Base 根目录的 config.py + .env；graph.py 会自己把根目录插进 sys.path
      3. 本目录的 .env.example 只放 docker-compose 要用的「容器/端口」变量
         （compose 只读同目录的 .env），其余 key 一个都不重复定义

    ---- 课案原样的 conf.py（只打印对照，本项目不生成、不使用）----
    # -*- coding: utf-8 -*-
    """【课案原文对照 · 本仓库不采用】课案原样的 conf.py
    ===============================================================
    课案的部署项目里，配置放在项目根的 conf.py，由 graph.py 去 import。
    
    ⚠️ 本项目**不用**这个文件，统一用 Python_Base 根目录的 config.py：
           from config import settings
       理由：全仓库一套配置来源，避免每个子项目各自维护一份 .env 和 Settings 类。
    
    另外课案 my_agent/graph.py 里写的是 `from config import setting`，
    和这个文件对不上（文件名是 conf.py，类实例叫 settings），是课案笔误。
    
    import os
    from pydantic import ConfigDict
    from pydantic_settings import BaseSettings
    
    PATH = os.path.dirname(os.path.abspath(__file__))
    
    
    class Settings(BaseSettings):
        api_key: str
        model_name: str
        base_url: str
    
        mysql_user: str
        mysql_password: str
        mysql_database: str
        mysql_host: str
        mysql_port: int
        baidu_qfan_api_key: str
        gitee_api_key: str
        dashscope_api_key: str
        dashscope_base_url: str
        langfuse_secret_key: str
        langfuse_public_key: str
        langfuse_host: str
    
        model_config = ConfigDict(
            extra="allow",
            env_file=f"{PATH}/.env",
            case_sensitive=False,
        )
    
    
    settings = Settings()
    """
```

## 9. `aegra.json`：本节最值钱的一条

课案原文就 6 行：

```json
{
  "graphs": {
    "agent": "./my_agent/graph.py:graph"
  },
  "dependencies": ["./"]
}
```

| 字段 | 说明 |
|---|---|
| `graphs` | `"agent名称": "文件路径:变量名"` 映射，与 `langgraph.json` 相同 |
| `dependencies` | **语义与 `langgraph.json` 完全不同**：这里是**加入 `sys.path` 的目录列表** |

⭐ 最容易抄错的就是 `dependencies`：

| 配置文件 | `dependencies` 的语义 |
|---|---|
| `langgraph.json` | 依赖包列表（pip 语义，会去装包） |
| `aegra.json` | **加入 `sys.path` 的目录列表**（不装任何东西） |

所以 `"dependencies": ["./"]` 是把**项目根目录塞进 `sys.path`**，
目的是让 `from config import settings` 这种「同项目内互相 import」能生效 ——
**不是**因为依赖装好了。

两个补充：

- `graphs` 里的 `"agent"` 就是客户端调用时的 `assistant_id`：启动时每个 graph 会自动注册
  一个**同名默认 assistant**，所以客户端直接填这个 key 就行（下一课 `04_客户端调用` 用到）。
  ⚠️ 课案「调用」那段的 `assistant_id` 写的是 `"bushu"`，那是课案作者自己的项目名，
  它**必须和这里的 key 一致**，否则会 404；
- `aegra.json` 是**纯 JSON，不支持注释**，所以下面生成的文件里一个 `#` 都不会有。

In [ ]:
# ================================================================
# 5. aegra.json 配置与字段说明
# ================================================================
AEGRA_JSON = """{
  "graphs": {
    "agent": "./my_agent/graph.py:graph"
  },
  "dependencies": ["./"]
}"""


def section_5_aegra_json() -> None:
    print("\n" + "=" * 78)
    print("5. aegra.json：配置与字段说明")
    print("=" * 78)
    for ln in AEGRA_JSON.splitlines():
        print("    " + ln)

    print_table(
        "aegra.json 字段说明",
        ["字段", "说明"],
        [
            ["graphs", "\"agent名称\": \"文件路径:变量名\" 映射，与 langgraph.json 相同"],
            ["dependencies",
             "语义与 langgraph.json 不同：这里是加入 sys.path 的目录列表（\"./\" = 项目根）"],
        ],
    )

    print("\n  ◆ graphs 里的 \"agent\" 就是 assistant_id")
    print("      启动时每个 graph 会自动注册一个**同名默认 assistant**，")
    print("      所以客户端调用时 assistant_id 直接填这个 key 就行（下一节 04 用到）。")
    print("      ⚠️ 课案「调用」那段的 assistant_id 写的是 \"bushu\"，那是课案作者自己的项目名，")
    print("         它必须和这里的 key 一致，否则会 404。")
    print("\n  ◆ dependencies 的语义差异（本节最值钱的一条）")
    print("      langgraph.json 的 dependencies 是**pip 依赖列表**（要去装包）；")
    print("      aegra.json 的 dependencies 是**加进 sys.path 的目录列表**（什么都不装）。")
    print("      所以 [\"./\"] 的作用是：让项目根目录下的模块能被 import。")
    print("      这就是 graph.py 里 `from config import settings` 能跑通的原因 ——")
    print("      **不是**因为依赖装好了，而是因为项目根被塞进了 sys.path。")
    print("\n      （顺带提醒：aegra.json 是纯 JSON，**不支持注释**，")
    print("        所以本节生成的文件里它一个 # 都没有。）")


section_5_aegra_json()

### 预期输出

```text

==============================================================================
5. aegra.json：配置与字段说明
==============================================================================
    {
      "graphs": {
        "agent": "./my_agent/graph.py:graph"
      },
      "dependencies": ["./"]
    }

【aegra.json 字段说明】
+--------------+-----------------------------------------------------------------------------+
| 字段         | 说明                                                                        |
+--------------+-----------------------------------------------------------------------------+
| graphs       | "agent名称": "文件路径:变量名" 映射，与 langgraph.json 相同                 |
| dependencies | 语义与 langgraph.json 不同：这里是加入 sys.path 的目录列表（"./" = 项目根） |
+--------------+-----------------------------------------------------------------------------+

  ◆ graphs 里的 "agent" 就是 assistant_id
      启动时每个 graph 会自动注册一个**同名默认 assistant**，
      所以客户端调用时 assistant_id 直接填这个 key 就行（下一节 04 用到）。
      ⚠️ 课案「调用」那段的 assistant_id 写的是 "bushu"，那是课案作者自己的项目名，
         它必须和这里的 key 一致，否则会 404。

  ◆ dependencies 的语义差异（本节最值钱的一条）
      langgraph.json 的 dependencies 是**pip 依赖列表**（要去装包）；
      aegra.json 的 dependencies 是**加进 sys.path 的目录列表**（什么都不装）。
      所以 ["./"] 的作用是：让项目根目录下的模块能被 import。
      这就是 graph.py 里 `from config import settings` 能跑通的原因 ——
      **不是**因为依赖装好了，而是因为项目根被塞进了 sys.path。

      （顺带提醒：aegra.json 是纯 JSON，**不支持注释**，
        所以本节生成的文件里它一个 # 都没有。）
```

## 10. 骨架里的七个文件，逐个看

第 6 节开始是「真正要落盘的东西」。每个文件的正文都放在一个 `dict` 里
（键 = 相对路径，值 = 文件内容），落盘循环只做三件事：拼路径 → 建父目录 → `write_text`。

| # | 文件 | 角色 |
|---|---|---|
| 1 | `aegra.json` | 部署配置。Aegra 启动时读它，知道「有哪些 graph、从哪 import」 |
| 2 | `requirements.txt` | 依赖清单，进 Docker 镜像时被 `pip install -r` 消费 |
| 3 | `Dockerfile` | 生产镜像配方，`aegra up` 构建应用容器时用它 |
| 4 | `.env.example` | 容器/端口变量的**模板**（刻意的，真 `.env` 不进仓库） |
| 5 | `docker-compose.yml` | `aegra dev` / `up` 内部调的就是 docker compose |
| 6 | `my_agent/__init__.py` | 空包标记，让 `my_agent` 成为可 import 的包 |
| 7 | `my_agent/graph.py` | 你的 Graph 定义，`aegra.json` 指向的就是它的模块级变量 `graph` |

⚠️ 刻意**没有** `conf.py`：见第 8 节的三处偏差。

### 10.1 `my_agent/graph.py` —— 骨架里唯一有「业务」的文件

三处值得停一下：

- `aegra.json` 写的是 `"./my_agent/graph.py:graph"`，即「文件路径 **:** 模块级变量名」，
  所以下面那个 `graph = ...` **必须叫 `graph`**；
- 配置**不另起一套**：直接用 Python_Base 根目录那份 `config.py` / `.env`。
  文件里的循环会向上找到第一个同时含 `config.py` + `pyproject.toml` 的目录（就是仓库根），
  插进 `sys.path`，`from config import settings` 才能生效
  —— 注意这一句**正是** `aegra.json` 那个 `dependencies` 干的活，只是它只加**项目根**，
  够不到 Python_Base 根目录，所以这里要自己补一句；
- `chatbot` 只把**最近 6 条**消息丢给模型：`MessagesState` 是无限增长的，
  全量传进去 token 会越来越贵，这里用最朴素的「滑动窗口记忆」。

In [ ]:
# ================================================================
# 6. 骨架文件内容（本节真正要落盘的东西）
# ================================================================
GRAPH_PY = '''# -*- coding: utf-8 -*-
"""
Aegra 项目的 Graph 定义（由 09_aegra_deploy/02_项目骨架_jxsd.py 生成）
================================================================
aegra.json 的 graphs 字段指向这里："./my_agent/graph.py:graph"
即「文件路径 : 模块级变量名」，所以下面那个 `graph = ...` 是**必须叫 graph** 的。

配置**不另起一套**：直接用 Python_Base 根目录那份 config.py / .env。
下面的循环会从本文件往上找，第一个同时含 config.py + pyproject.toml 的目录
就是 Python_Base 根目录，把它插进 sys.path，`from config import settings` 才能生效。
（aegra.json 的 "dependencies": ["./"] 只把**本项目**根目录加进 sys.path，
够不到 Python_Base 根目录，所以这里要自己补一句。这就是那个字段的真实作用。）

⚠️ 放到服务器上独立部署时，Python_Base 根目录不在场，按课案原样做即可：
   把根目录的 config.py 和 .env 一起带过去（**仍然只有这一份配置**），不用另写 conf.py。
"""

import sys
from pathlib import Path

# 向上找到 Python_Base 根目录（含 config.py + pyproject.toml 的那一层）
for _p in Path(__file__).resolve().parents:
    if (_p / "config.py").exists() and (_p / "pyproject.toml").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessagesState, StateGraph

from config import settings  # 根目录统一配置，不另建一套

# 课案用的是 ChatOpenAI 直连写法（部署项目自成一体，不依赖 Python_Base 的
# init_chat_model 封装），但三个参数一律来自 settings，绝不硬编码。
llm = ChatOpenAI(
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


def chatbot(state: MessagesState):
    """最简单的单节点对话：把最近 6 条消息丢给模型

    为什么只取 [-6:]：MessagesState 是无限增长的，全量传进去 token 会越来越贵。
    这里截最近 6 条做上下文窗口，是课案里最朴素的「滑动窗口记忆」写法。
    （真正要长期记忆时，应换成 checkpoint + 摘要，见 01_langgraph 那几节。）
    """
    return {"messages": [llm.invoke(state["messages"][-6:])]}


# 图的组装：一个节点、两条边，START → chatbot → END
# 用链式写法（课案原样）而不是 builder.add_node 逐行写。
graph = (
    StateGraph(MessagesState)
    .add_node("chatbot", chatbot)
    .add_edge(START, "chatbot")
    .add_edge("chatbot", END)
    .compile()
)
'''

### 10.2 `my_agent/__init__.py` —— 一个空包标记

有 `__init__.py`，`my_agent` 才是一个可 import 的**包**，
Aegra 才能按 `aegra.json` 里的 `"./my_agent/graph.py:graph"` 找到图。

内容保持为空即可：**不要**在这里 `import graph` —— 那会让项目启动时多绕一层。

In [ ]:
INIT_PY = '''# -*- coding: utf-8 -*-
"""my_agent 包标记（由 09_aegra_deploy/02_项目骨架_jxsd.py 生成）

有 __init__.py，my_agent 才是一个可 import 的包，
Aegra 才能按 aegra.json 里的 "./my_agent/graph.py:graph" 找到图。
内容保持为空即可，不要在这里 import graph —— 那会让项目启动时多绕一层。
"""
'''

### 10.3 `requirements.txt` —— 依赖清单

课案原文这 8 行包名，进 Docker 镜像时被 `pip install -r requirements.txt` 消费。
各自的作用（`deepagents` / `langfuse` 等）在下一课的常量说明里再展开。

In [ ]:
REQUIREMENTS_TXT = '''# Aegra 项目依赖（由 09_aegra_deploy/02_项目骨架_jxsd.py 生成）
# 课案原文这 8 行包名，各自作用见 05_对接Langfuse_jxsd.py 里的常量说明。
aegra-cli
aegra-api
langgraph
langchain
langchain-openai
deepagents
langfuse
pydantic-settings
'''

### 10.4 `Dockerfile` —— 生产镜像配方（`aegra up` 用它构建应用容器）

三个要点：基础镜像是 `python:3.12-slim`；`PIP_INDEX_URL` 换成清华源加速国内拉包；
最后 `EXPOSE 2026` + `CMD ["aegra", "serve", ...]`
—— 注意 **`aegra serve` 在 Windows 上跑不了**（见 7.4），这个命令是给 Linux 容器用的。

In [ ]:
DOCKERFILE = '''# 生产镜像（由 09_aegra_deploy/02_项目骨架_jxsd.py 生成）
# aegra up 会用这个文件构建应用容器；逐行中文注释见 03_本地开发与生产部署_jxsd.py
FROM python:3.12-slim

WORKDIR /app

# 国内 PyPI 镜像加速
ENV PIP_INDEX_URL=https://pypi.tuna.tsinghua.edu.cn/simple

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 2026
CMD ["aegra", "serve", "--host", "0.0.0.0", "--port", "2026"]
'''

### 10.5 `.env.example` —— 容器/端口变量的模板

⚠️ 这里**不是**第二套配置：大模型、数据库这些应用配置全仓库只有一份来源
（Python_Base 根目录的 `.env`）。本文件只负责 **docker-compose 需要的容器/端口变量**
（compose 只读同目录的 `.env`），其余 key 一个都不重复定义。

用的时候 `cp .env.example .env` 再填真值；**模板里全是占位符**（`change-me`），
任何真实口令都不要写进模板、不要提交进仓库。

In [ ]:
ENV_EXAMPLE = '''# Aegra 项目的 .env 模板（由 09_aegra_deploy/02_项目骨架_jxsd.py 生成）
#
# ⚠️ 这里**不是**第二套配置！大模型、数据库这些应用配置全仓库只有一份来源：
#       Python_Base 根目录的  F:\\ProGram\\Python_Base\\.env
#   本文件只负责 docker-compose 需要的「容器 / 端口」变量（compose 只读同目录的 .env），
#   其余 key 一个都不重复定义。
#
# 用法：cp .env.example .env   （.env 不进 Git；模板里全是占位符，不要填真值进模板）

# ---------- docker-compose 用：PostgreSQL 容器 ----------
POSTGRES_USER="my_agent_project"
POSTGRES_PASSWORD="change-me"
POSTGRES_DB="my_agent_project"
POSTGRES_HOST="localhost"
POSTGRES_PORT=5434
PORT=2026

# ---------- Langfuse 追踪开关（见 05_对接Langfuse_jxsd.py） ----------
# 这四行一加，代码零改动就能开启追踪；Aegra 通过 OTEL_TARGETS 决定往哪推 span。
# 真实 pk/sk 写在 Python_Base 根目录 .env 或本目录 .env，别写进模板。
OTEL_TARGETS="LANGFUSE"
LANGFUSE_BASE_URL="http://localhost:3000"
'''

### 10.6 `docker-compose.yml` —— 三个服务（78 行，拆成两格续写）

结构逐字保留课案 405-487 行那份，唯一改动是**口令类默认值换成 `change-me` 占位符**
（用户名/库名 `my_agent_project` 不是口令，原样保留）。

三个服务：

| 服务 | 镜像 | 什么时候起 |
|---|---|---|
| `postgres` | `pgvector/pgvector:pg18` | `aegra dev`（只起数据库） |
| `redis` | `redis:7-alpine` | `aegra up`（生产全栈，Redis 当 broker） |
| `my_agent_project` | 用同目录 `Dockerfile` 构建 | `aegra up` |

先写第一段（头部注释 + `postgres` + `redis`）：

In [ ]:
# docker-compose.yml —— 课案 405-487 行那份，结构逐字保留。
# 唯一的改动：口令类默认值换成占位符 change-me（任何像口令的字符串都不进仓库）。
DOCKER_COMPOSE_YML = '''# Docker Compose - PostgreSQL + Redis + API
# aegra dev  -> docker compose up postgres -d  (database only, in-memory broker)
# aegra up   -> docker compose up --build      (full stack, Redis broker)
# （由 09_aegra_deploy/02_项目骨架_jxsd.py 生成；口令默认值已改为 change-me 占位符）

services:
  postgres:
    image: pgvector/pgvector:pg18
    container_name: my_agent_project-postgres
    restart: unless-stopped
    environment:
      POSTGRES_USER: ${POSTGRES_USER:-my_agent_project}
      POSTGRES_PASSWORD: ${POSTGRES_PASSWORD:-change-me}
      POSTGRES_DB: ${POSTGRES_DB:-my_agent_project}
    ports:
      - "${POSTGRES_PORT:-5434}:5432"
    volumes:
      - postgres_data:/var/lib/postgresql
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U ${POSTGRES_USER:-my_agent_project}"]
      interval: 5s
      timeout: 5s
      retries: 5

  redis:
    image: redis:7-alpine
    container_name: my_agent_project-redis
    restart: unless-stopped
    ports:
      - "6380:6379"
    volumes:
      - redis_data:/data
    healthcheck:
      test: ["CMD", "redis-cli", "ping"]
      interval: 5s
      timeout: 5s
      retries: 5
'''

第二段：应用服务本体 + 卷声明。

注意四个环境变量是在哪里生效的：

- `AEGRA_CONFIG=aegra.json` —— 告诉服务读哪个配置文件；
- `POSTGRES_HOST=postgres` / `POSTGRES_PORT=5432` —— **容器内**的主机名与端口
  （不是宿主机的 `localhost:5434`，这是容器网络里的地址）；
- `REDIS_URL=redis://redis:6379/0` —— 同理，走 compose 内的服务名；
- `LANGFUSE_BASE_URL=http://host.docker.internal:3000` —— 容器里的 `localhost`
  指向容器自己，要访问宿主机上的 Langfuse 必须走 `host.docker.internal`。

In [ ]:
DOCKER_COMPOSE_YML += '''
  my_agent_project:
    build: .
    container_name: my_agent_project-api
    restart: unless-stopped
    ports:
      - "${PORT:-2026}:${PORT:-2026}"
    env_file:
      - .env
    environment:
      - POSTGRES_USER=${POSTGRES_USER:-my_agent_project}
      - POSTGRES_PASSWORD=${POSTGRES_PASSWORD:-change-me}
      - POSTGRES_HOST=postgres
      - POSTGRES_PORT=5432
      - POSTGRES_DB=${POSTGRES_DB:-my_agent_project}
      - AEGRA_CONFIG=aegra.json
      - AUTH_TYPE=${AUTH_TYPE:-noop}
      - PORT=${PORT:-2026}
      - REDIS_BROKER_ENABLED=true
      - REDIS_URL=redis://redis:6379/0
      # 启用 Langfuse 追踪；容器内 localhost 指向自身，需走宿主机网关访问 langfuse
      - OTEL_TARGETS=LANGFUSE
      - LANGFUSE_BASE_URL=http://host.docker.internal:3000
    depends_on:
      postgres:
        condition: service_healthy
      redis:
        condition: service_healthy
    healthcheck:
      test: ["CMD-SHELL", "curl -sf http://localhost:${PORT:-2026}/health || exit 1"]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 10s
    volumes:
      - ./src:/app/src:ro
      - ./aegra.json:/app/aegra.json:ro

volumes:
  postgres_data:
  redis_data:
'''

### 10.7 把七个文件装进 `SKELETON_FILES`

键是**相对 `aegra_project/` 的路径**，值就是上面写好的文件正文。
全部内容集中在这一个 `dict` 里，所以想改骨架内容只改它的值即可 ——
落盘循环（下一节）只负责拼路径、建目录、写字节。

两个细节：

- `aegra.json` 的值是 `AEGRA_JSON + "\n"`：纯 JSON 不支持注释，这里只补一个结尾换行；
- 文件里**没有** `conf.py`（见第 8 节）。

In [ ]:
# 骨架文件清单：相对路径 → 内容。
# 键是【相对 aegra_project/ 的路径】，值就是上面那几个常量里写好的文件正文。
SKELETON_FILES = {
    "aegra.json": AEGRA_JSON + "\n",
    "requirements.txt": REQUIREMENTS_TXT,
    "Dockerfile": DOCKERFILE,
    ".env.example": ENV_EXAMPLE,
    "docker-compose.yml": DOCKER_COMPOSE_YML,
    "my_agent/__init__.py": INIT_PY,
    "my_agent/graph.py": GRAPH_PY,
}

print(f"骨架文件清单：{len(SKELETON_FILES)} 个")
for _rel, _content in SKELETON_FILES.items():
    print(f"    {_rel:<22} {len(_content.splitlines()):>3} 行  {len(_content.encode('utf-8')):>5} 字节")

### 预期输出

```text
骨架文件清单：7 个
    aegra.json               6 行     89 字节
    requirements.txt        10 行    268 字节
    Dockerfile              16 行    466 字节
    .env.example            22 行   1088 字节
    docker-compose.yml      78 行   2362 字节
    my_agent/__init__.py     7 行    345 字节
    my_agent/graph.py       60 行   2662 字节
```

最后一列字节数与仓库里那份真实骨架**完全一致** —— 说明本课生成的骨架和已入库的那份
是同一批内容（`docker-compose.yml` 的 2362 字节里不含任何真实口令）。

## 11. 真的落盘：写入 → 打印目录树 → 冒烟导入

### 11.1 把七个文件写到 `WORKDIR / "aegra_skeleton"`

落盘循环里两个「为什么」：

- **`newline="\n"`**：保证 LF 换行。`Dockerfile` / `docker-compose.yml` 要进 Linux 容器，
  Windows 默认写成 CRLF 时，某些镜像里的 shell 解析会出莫名其妙的错；
- **`.env` 只生成 `.env.example`**：真 `.env` 里是真实密钥，进仓库就漏了。
  ⚠️ 但它**不是可选的**：`docker-compose.yml` 里写了 `env_file: - .env`，
  缺了它连 `docker compose config` 都会直接报 `env file ...\.env not found`。

应用配置（`API_KEY` / `MODEL_NAME` / `PG_URI` …）**不在**这里配：
全仓库统一读 Python_Base 根目录的 `.env`。

In [ ]:
def section_6_generate() -> None:
    """真的把骨架写到硬盘上"""
    print("\n" + "=" * 78)
    print("6. 生成项目骨架（真的落盘）")
    print("=" * 78)
    print(f"  目标目录：{PROJECT_DIR}")

    for rel, content in SKELETON_FILES.items():
        path = PROJECT_DIR / rel
        path.parent.mkdir(parents=True, exist_ok=True)   # my_agent/ 需要自动创建
        # newline="\n" 保证 LF 换行：Dockerfile / docker-compose.yml 进 Linux 容器更稳
        path.write_text(content, encoding="utf-8", newline="\n")
        print(f"    [已写入] {rel:<24} {path.stat().st_size:>5} 字节")

    print("\n  ⚠️ 只生成了 .env.example，**没有**生成 .env ——")
    print("     真 .env 里是真实密钥，进仓库就漏了；用的时候 cp .env.example .env 再填。")
    print("     ⚠️ 这个 .env 不是可选的：docker-compose.yml 里写了 `env_file: - .env`，")
    print("        缺了它连 `docker compose config` 都会直接报")
    print("        「env file ...\\.env not found」，服务更起不来。")
    print("     实测：cp .env.example .env 之后 `docker compose config --quiet` 退出码为 0，")
    print("           说明生成的 docker-compose.yml 与 .env.example 是彼此匹配的。")
    print("\n  ⚠️ 应用配置（API_KEY / MODEL_NAME / PG_URI ...）**不在**这里配：")
    print("     全仓库统一读 Python_Base 根目录的  F:\\ProGram\\Python_Base\\.env")
    print("     graph.py 会向上找到含 config.py 的那层插进 sys.path，再 from config import settings。")


section_6_generate()

### 预期输出

```text

==============================================================================
6. 生成项目骨架（真的落盘）
==============================================================================
  目标目录：F:\ProGram\Python_Base\Agent\09_aegra_deploy\tmp_nb_work\aegra_skeleton
    [已写入] aegra.json                  89 字节
    [已写入] requirements.txt           268 字节
    [已写入] Dockerfile                 466 字节
    [已写入] .env.example              1088 字节
    [已写入] docker-compose.yml        2362 字节
    [已写入] my_agent/__init__.py       345 字节
    [已写入] my_agent/graph.py         2662 字节

  ⚠️ 只生成了 .env.example，**没有**生成 .env ——
     真 .env 里是真实密钥，进仓库就漏了；用的时候 cp .env.example .env 再填。
     ⚠️ 这个 .env 不是可选的：docker-compose.yml 里写了 `env_file: - .env`，
        缺了它连 `docker compose config` 都会直接报
        「env file ...\.env not found」，服务更起不来。
     实测：cp .env.example .env 之后 `docker compose config --quiet` 退出码为 0，
           说明生成的 docker-compose.yml 与 .env.example 是彼此匹配的。

  ⚠️ 应用配置（API_KEY / MODEL_NAME / PG_URI ...）**不在**这里配：
     全仓库统一读 Python_Base 根目录的  F:\ProGram\Python_Base\.env
     graph.py 会向上找到含 config.py 的那层插进 sys.path，再 from config import settings。
```

（`目标目录` 一行随运行环境变化：notebook 里是本课临时目录 `tmp_nb_work/aegra_skeleton`，
源脚本里则是脚本同级的 `aegra_project`。）

### 11.2 打印目录树 + 冒烟导入

这一段是「真的把生成的 `graph.py` import 进来」，确认它不是一个坏的骨架。

`walk` 函数两个设计点：

- 跳过 `__pycache__`：那是跑出来的字节码缓存，不属于骨架的组成部分；
- 排序键 `(p.is_file(), p.name)`：目录排在文件前面、同类按名字排序。
  不排序的话 `iterdir()` 的顺序由文件系统决定，每次打印的树都不一样。

冒烟导入前手工模拟 `aegra.json` 的 `"dependencies": ["./"]` 效果：
把**项目根**（找 `my_agent` 包）与**仓库根**（找 `config` 模块）都塞进 `sys.path`。
导入完还要 `del sys.modules[...]`，免得后续课时拿到这份缓存的模块。

In [ ]:
def section_7_tree_and_smoke() -> None:
    """打印生成后的目录树，并对生成的 graph.py 做一次冒烟导入"""
    print("\n" + "=" * 78)
    print("7. 生成结果：目录树 + 冒烟校验")
    print("=" * 78)
    print(f"{PROJECT_DIR.name}/")

    def walk(directory: Path, prefix: str = "") -> None:
        # 跳过 __pycache__：那是跑出来的字节码缓存，不属于骨架的组成部分
        entries = sorted(
            (p for p in directory.iterdir() if p.name != "__pycache__"),
            # 排序键 (is_file, name)：目录排在文件前面，同类按名字排序。
            # 不排序的话 iterdir() 的顺序由文件系统决定，每次打印的树都不一样。
            key=lambda p: (p.is_file(), p.name),
        )
        for i, entry in enumerate(entries):
            last = i == len(entries) - 1
            branch = "└── " if last else "├── "
            size = f"   # {entry.stat().st_size} 字节" if entry.is_file() else "/"
            print(f"{prefix}{branch}{entry.name}{size}")
            if entry.is_dir():
                walk(entry, prefix + ("    " if last else "│   "))

    walk(PROJECT_DIR)

    # ---- 冒烟校验：真的把生成的 graph.py import 进来 ----
    # aegra.json 的 "dependencies": ["./"] 会把项目根加进 sys.path，
    # 这里手工模拟同样的效果：项目根（找 my_agent 包） + 仓库根（找 config 模块）。
    print("\n  [冒烟校验] 尝试 import 生成的 my_agent.graph ...")
    for p in (str(REPO_ROOT), str(PROJECT_DIR)):
        if p not in sys.path:
            sys.path.insert(0, p)
    try:
        module = importlib.import_module("my_agent.graph")
        graph = module.graph
        nodes = sorted(graph.get_graph().nodes.keys())
        print(f"    ✅ import 成功，模块文件：{module.__file__}")
        print(f"    ✅ graph 对象：{type(graph).__name__}，节点：{nodes}")
        del sys.modules["my_agent.graph"]
        del sys.modules["my_agent"]
    except Exception as exc:   # noqa: BLE001 —— 骨架是生成物，坏掉必须看得见原因
        print(f"    ❌ 导入失败：{type(exc).__name__}: {exc}")
        print("       常见原因：仓库根目录的 config.py / .env 不完整，或缺 langchain-openai。")
        return

    print("\n  [说明] 当前骨架里的 graph.py 用的是 Python_Base 根目录的 config.py，")
    print("         所以在这里能 import 成功。真正部署到服务器时，把 Python_Base 根目录的")
    print("         config.py + .env 一起带过去（仍然只有这一份配置），不用另写 conf.py。")

    # 清掉刚才 import 产生的字节码缓存，让生成物目录保持干净
    removed = 0
    for cache in PROJECT_DIR.rglob("__pycache__"):
        shutil.rmtree(cache, ignore_errors=True)
        removed += 1
    if removed:
        print(f"\n  [清理] 已删除 {removed} 个 __pycache__（冒烟导入的副产物）")


section_7_tree_and_smoke()

### 预期输出

```text

==============================================================================
7. 生成结果：目录树 + 冒烟校验
==============================================================================
aegra_skeleton/
├── my_agent/
│   ├── __init__.py   # 345 字节
│   └── graph.py   # 2662 字节
├── .env.example   # 1088 字节
├── Dockerfile   # 466 字节
├── aegra.json   # 89 字节
├── docker-compose.yml   # 2362 字节
└── requirements.txt   # 268 字节

  [冒烟校验] 尝试 import 生成的 my_agent.graph ...
    ✅ import 成功，模块文件：F:\ProGram\Python_Base\Agent\09_aegra_deploy\tmp_nb_work\aegra_skeleton\my_agent\graph.py
    ✅ graph 对象：CompiledStateGraph，节点：['__end__', '__start__', 'chatbot']

  [说明] 当前骨架里的 graph.py 用的是 Python_Base 根目录的 config.py，
         所以在这里能 import 成功。真正部署到服务器时，把 Python_Base 根目录的
         config.py + .env 一起带过去（仍然只有这一份配置），不用另写 conf.py。

  [清理] 已删除 1 个 __pycache__（冒烟导入的副产物）
```

⭐ 这一格里最值得看的是冒烟导入那两行：

- `import 成功` 说明**骨架是完整的**：`aegra.json` 指向的 `my_agent/graph.py:graph`
  真的能 import 出来一个模块级变量 `graph`；
- `节点：['__start__', 'chatbot', '__end__']` 说明这个对象**真的是一个编译好的 LangGraph**
  —— `__start__` / `__end__` 是框架插入的虚拟节点，`chatbot` 才是我们写的那个。

> 本机没有 `aegra` 命令也不影响这一格：冒烟导入只用到 `langgraph` + `langchain-openai`
> + 仓库根的 `config.py`，**不连模型、不起服务**。

## 12. 仓库里那份真实骨架（只读展示）

上面第 11 节写的是本课临时目录里的副本。仓库里 `Agent/09_aegra_deploy/aegra_project/`
下还有一份**已入库**的真实骨架 —— 本课只读它，一个字都不动。

这里的遍历不复用上一格的 `walk`（它是 `section_7_tree_and_smoke` 的**局部函数**，
拿到外面来会把那段代码源文件的行号结构搞乱），改用更紧凑的一行式列出相对路径 + 字节数。

In [ ]:
print(f"{REAL_PROJECT_DIR.relative_to(NB_DIR).as_posix()}/   （已入库，本课只读）")
for _p in sorted(REAL_PROJECT_DIR.rglob("*")):
    if "__pycache__" in _p.parts:
        continue
    _rel = _p.relative_to(REAL_PROJECT_DIR).as_posix()
    _size = "" if _p.is_dir() else f"   # {_p.stat().st_size} 字节"
    print(f"    {_rel:<24}{_size}")

print("\n  说明：这份骨架就是上面第 6 节的生成逻辑跑出来的结果，")
print("        下一课的 `aegra dev` 直接在这个目录里执行。")

### 预期输出

```text
aegra_project/   （已入库，本课只读）
    .env.example               # 1088 字节
    aegra.json                 # 89 字节
    docker-compose.yml         # 2362 字节
    Dockerfile                 # 466 字节
    my_agent                
    my_agent/__init__.py       # 345 字节
    my_agent/graph.py          # 2662 字节
    requirements.txt           # 268 字节

  说明：这份骨架就是上面第 6 节的生成逻辑跑出来的结果，
        下一课的 `aegra dev` 直接在这个目录里执行。
```

⚠️ 三处细节：

- `rglob("*")` 是**平铺**列出，不是树：`my_agent/__init__.py`、`my_agent/graph.py`
  紧跟在 `my_agent` 那一行之后（顺序是先目录后文件、同类按名字排序）；
- 目录行**没有字节数**（`my_agent` 那行末尾的空白只是列对齐补齐），文件行才有；
- 这份列表与上面第 11 节生成的临时骨架**逐字节一致**
  （89 / 268 / 466 / 1088 / 2362 / 345 / 2662），说明仓库里这份就是同一段生成逻辑跑出来的。

> 想核对完整清单：`git ls-files Agent/09_aegra_deploy/aegra_project`。

## 13. 下一步：三条命令

骨架有了，接下来就是在它里面把服务起起来 —— 这一步在下一课。

In [ ]:
print(f"\n下一步：cd {PROJECT_DIR}")
print("        cp .env.example .env   # 填上真实值（.env 不进 Git）")
print("        uv run aegra dev       # 本地开发（详见 03_本地开发与生产部署_jxsd.py）")

### 预期输出

```text

下一步：cd F:\ProGram\Python_Base\Agent\09_aegra_deploy\tmp_nb_work\aegra_skeleton
        cp .env.example .env   # 填上真实值（.env 不进 Git）
        uv run aegra dev       # 本地开发（详见 03_本地开发与生产部署_jxsd.py）
```

## 小结

**上半节（为什么）**

- `langgraph build` 出的官方 `langgraph-api` 镜像**要 license**，启动即退出，
  报 `ValueError: License verification failed...` —— 这不是你的图写错了；
- **库免费、平台收费**：`langgraph` / `langgraph-checkpoint-*` / `langgraph-sdk`
  都是 MIT；只有官方那层「部署运行时」是商业产品；
- LangSmith 一个产品干两件事，可替代性完全不同：**追踪能用 Langfuse 顶掉，
  运行时 Langfuse 根本不做**，那块要靠 Aegra（或自己用 FastAPI 包一层）；
- 运行时四要素（Threads API / Runs / 流式 / Checkpoint 管理）在「用库」形态下
  **零件都在但得自己拼**，在「用平台」形态下被封装成一套 REST API；
- Aegra：Apache 2.0，同一套 Agent Protocol，`langgraph_sdk` 客户端原样可用，无 license 校验。

**下半节（长什么样）**

- 前置条件：Python 3.11+ / Docker / Redis，后两个由 Aegra **自动拉起**（但仍需 Docker 在运行）；
- 五个命令：`init` / `dev` / `up` / `down` / `serve`，
  其中 **`aegra serve` 在 Windows 上跑不了**，本地开发一律用 `aegra dev`；
- 骨架七文件：`aegra.json` + `requirements.txt` + `Dockerfile` + `.env.example`
  + `docker-compose.yml` + `my_agent/__init__.py` + `my_agent/graph.py`；
- ⭐ `aegra.json` 的 `dependencies` 是**加入 `sys.path` 的目录列表**，
  和 `langgraph.json` 的「pip 依赖列表」完全是两回事 —— 这是抄配置最容易踩的一条；
- 冒烟导入通过 = 骨架可用：`my_agent/graph.py` 真能 import 出一个编译好的 `graph`。

## 常见坑

1. **把 license 报错当成自己的 bug**：`ValueError: License verification failed...`
   是官方镜像的门槛，不是图写错了。要么配 license，要么换 Aegra。
2. **以为上了 Langfuse 就不需要部署运行时了**：Langfuse 是观测平台，
   threads/runs/流式/checkpoint 这一层它完全不做。
3. **抄 `dependencies` 时按 pip 语义理解**：`aegra.json` 里它是 `sys.path` 目录列表，
   `["./"]` 只是「把项目根加进 `sys.path`」，不会装任何包。
4. **`aegra.json` 里写注释**：它是纯 JSON，不支持 `#` 注释，加了会解析失败。
5. **`assistant_id` 和 `graphs` 的 key 不一致**：启动时会按 key 自动注册同名默认
   assistant，客户端填错就是 404（课案里的 `"bushu"` 就是作者自己的项目名）。
6. **只生成 `.env.example` 就以为没事**：`docker-compose.yml` 里 `env_file: - .env`
   是硬依赖，缺 `.env` 连 `docker compose config` 都直接报 `env file ... not found`。
7. **在 Windows 上折腾 `aegra serve`**：它依赖 Unix 的进程/信号模型，
   换任何参数都跑不起来，本地开发用 `aegra dev`。
8. **把真实口令写进模板/仓库**：模板里只能是 `change-me` 这类占位符，
   真值放在本地那份 `.env`（已在 `.gitignore` 里）。

## 官方链接

- LangGraph 部署总览（部署形态与选项）：<https://docs.langchain.com/oss/python/langgraph/deploy>
- LangGraph 持久化 / Checkpoint（表二「Checkpoint 管理」那一行）：<https://docs.langchain.com/oss/python/langgraph/persistence>
- LangGraph 流式输出（表二「流式」那一行）：<https://docs.langchain.com/oss/python/langgraph/streaming>
- Aegra（LangGraph Platform 的开源自托管替代，Apache 2.0）：<https://github.com/aegra/aegra>
- Langfuse（观测/追踪的开源替代，MIT）：<https://langfuse.com/docs>